# Hands-On AI for Science
## August 14, Morning: Foundation Models Without a GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jhasegaw/hands_on_ai_for_science/blob/main/lectures/colab/a14am_foundation_colab.ipynb)

**No Python installed?** Click the badge above to open a self-contained version of this notebook in Google Colab.  It runs in your browser, with nothing to download and nothing to install.

Yesterday afternoon, an embedding space was explored that had been computed in advance.  This session removes the "in advance". The model itself gets downloaded, run on our own laptops, and used to extract representations of new data. This is the complete workflow, end to end.  The goal is a very reusable hour of the workshop: the same recipe works for proteins, DNA, cells, EEG and fMRI data, images, and audio, whenever someone else has already paid to train a model.

## Setup

The cell below checks packages and files.  **If anything prints MISSING, fix it now**: `pip install torch transformers` in a terminal (part of the pre-workshop setup), then **restart the kernel** and re-run from the top.  A MISSING file means Jupyter was started outside the repository's `lectures` folder.

The model itself is checked separately in Section 3 — and if it turns out to be unavailable (no cache, no wifi), **this notebook has a built-in fallback** and every section still runs.

In [1]:
import importlib.util, os

for pkg in ['torch', 'transformers', 'numpy', 'pandas', 'sklearn']:
    print(f'{pkg:28s}', 'OK' if importlib.util.find_spec(pkg) is not None else 'MISSING')
for f in ['protein_localization.csv', 'protein_embeddings.npy', 'a14am_hw2.py']:
    print(f'{f:28s}', 'OK' if os.path.exists(f) else 'MISSING')

torch                        OK
transformers                 OK
numpy                        OK
pandas                       OK
sklearn                      OK
protein_localization.csv     OK
protein_embeddings.npy       OK
a14am_hw2.py                 OK



1. [What a foundation model is](#what)
1. [Three ways to use one](#ways)
1. [Loading the model](#loading)
1. [Extracting representations](#extracting)
1. [The comparison: learned representation vs. simple baseline](#comparison)
1. [Per-residue embeddings and attention](#residue)
1. [Homework](#homework)
1. [Summary](#recipe)

<a id="what"></a>

## 1. What a foundation model is

One idea: a **foundation model** is a model trained *once*, at enormous expense, on huge amounts of **unlabeled** data, with a self-supervised objective — and then reused by everyone for tasks its trainers never imagined.

The training objective is usually some version of *predict a hidden piece from its context*.  For the model used today, ESM-2, the recipe was: take ~65 million protein sequences, and for each instance of a protein, hide random residues. Then, train the model to guess them from the surrounding residues.  That is word2vec's idea — the one built by hand yesterday — scaled up a few million times.  To get good at that guessing game, the model is forced to learn a great deal about how sequences work; the representations it builds along the way are the product we extract.

The same pattern now exists across scientific data types.  A partial family portrait:

| Data | Models |
|---|---|
| protein sequence | ESM, ProtTrans |
| DNA | Evo, Nucleotide Transformer |
| single-cell expression | scGPT, Geneformer |
| images (incl. microscopy) | SAM, DINOv2 |
| audio | Whisper |
| text | the LLMs everyone has seen |

The economics are the point: training ESM-2's larger versions cost more compute than most labs will ever have — and using the result costs a laptop.

<a id="ways"></a>

## 2. Three ways to use one

In increasing order of cost:

1. **Frozen feature extraction** — run data through the model, keep the internal representations, analyze them with the tools from this week (probes, distances, projections).  Costs nothing but a CPU; works with as few as ~50 labeled examples.
2. **A small trained head** — the linear probe, or something slightly bigger, trained on top of frozen representations.  Still cheap; this is what we have been doing.
3. **Full fine-tuning** — updating the model's own weights on labeled data.  Needs a GPU, hyperparameter judgment, and far more labeled data than most projects have.  (Cheaper variants exist — the names to recognize are **LoRA** and **PEFT** — but the judgment burden remains.)

**For most scientific projects, option 1 or 2 is the right answer, and fine-tuning is a trap**. This is not because fine tuning never helps, but because it usually isn't needed. It converts a week-long project with a laptop into a months-long engineering project.  The demonstration this hour is that option 1 already carries remarkable information.

<a id="loading"></a>

## 3. Loading the model

Pretrained models are distributed through hubs — the largest is [Hugging Face](https://huggingface.co) — and the `transformers` library downloads and loads them by name.  Two things to know before running the next cell:

* The model downloads **once** (~31 MB for ESM-2's smallest version) into a cache folder (`~/.cache/huggingface/`), and every later load is instant and offline.  The pre-workshop setup script did this download already.
* The load prints a scary-looking report about UNEXPECTED and MISSING keys.  **This is expected, not an error.**  The checkpoint includes the model's training-task head (the masked-residue guesser, `lm_head`), which is deliberately not loaded — the representations are what's wanted, not the guessing game.  A `pooler` is freshly created and never used; the pooling is done by hand below, where it can be seen.

If the model is unavailable (no cache and no wifi), the cell says so, and the notebook automatically falls back to precomputed embeddings — Sections 5 onward are identical either way.

In [2]:
import numpy as np
import pandas as pd
import torch

proteins = pd.read_csv('protein_localization.csv')
shipped_embeddings = np.load('protein_embeddings.npy')

try:
    from transformers import AutoModel, AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained('facebook/esm2_t6_8M_UR50D')
    model = AutoModel.from_pretrained('facebook/esm2_t6_8M_UR50D')
    model.eval()
    MODEL_AVAILABLE = True
    print('\nModel loaded:', sum(p.numel() for p in model.parameters()), 'parameters.')
except Exception as e:
    MODEL_AVAILABLE = False
    print('Model could not be loaded:', type(e).__name__)
    print('Falling back to the precomputed embeddings -- everything below still runs.')

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Model loaded: 7511801 parameters.


<a id="extracting"></a>

## 4. Extracting representations

Three steps.
1) **Tokenization**: the model reads integer token ids, not letters — the same idea as the addition model an hour ago, with an amino-acid vocabulary.
2) **Padding**: Because sequences have different lengths, batching them requires padding.
3) **Attention mask** — the exact structure implemented in the previous session's homework.  Watch them appear:

In [4]:
if MODEL_AVAILABLE:
    two = tokenizer(list(proteins['sequence'][:2]), padding=True, return_tensors='pt')
    print('sequence lengths:', [len(s) for s in proteins['sequence'][:2]])
    print('input_ids shape: ', tuple(two['input_ids'].shape), ' (padded to the longer one, +2 special tokens)')
    print('attention mask row sums:', two['attention_mask'].sum(dim=1).tolist(), ' <- real positions per sequence')
else:
    print('(model unavailable -- skipping the live demo)')

sequence lengths: [207, 282]
input_ids shape:  (2, 284)  (padded to the longer one, +2 special tokens)
attention mask row sums: [209, 284]  <- real positions per sequence


The **forward pass** produces `last_hidden_state`: one vector per token position, shape (batch, length, 320).  To get one vector per *sequence*, the position vectors are averaged — with one crucial detail. **Padding positions must be excluded from the average**, using the attention mask.  Averaging padding into a short sequence's embedding drags it toward zero. This is a mistake that produces no error, only worse results.  An alternative convention pools a special CLS token instead; mean-pooling is the common default for sequence-level work.

The full extraction, on all 450 proteins:

In [5]:
def extract(model, tokenizer, seqs, batch_size=16):
    out = []
    with torch.no_grad():
        for start in range(0, len(seqs), batch_size):
            batch = tokenizer(seqs[start:start+batch_size], padding=True, return_tensors='pt')
            hidden = model(**batch).last_hidden_state          # (b, L, 320)
            mask = batch['attention_mask'].unsqueeze(-1)       # (b, L, 1)
            out.append(((hidden * mask).sum(1) / mask.sum(1)).numpy())
    return np.concatenate(out).astype(np.float32)

import time
if MODEL_AVAILABLE:
    t0 = time.time()
    embeddings = extract(model, tokenizer, list(proteins['sequence']))
    print(f'{len(proteins)} proteins embedded in {time.time()-t0:.1f} s on CPU')
    print('matches the precomputed file:', np.allclose(embeddings, shipped_embeddings, atol=1e-4))
else:
    embeddings = shipped_embeddings
    print('using precomputed embeddings:', embeddings.shape)

450 proteins embedded in 4.4 s on CPU
matches the precomputed file: True


A few seconds for 450 proteins on my laptop. The result reproduces the file we used yesterday.  This is the cost of "using a foundation model" in extraction mode.  No GPU appeared at any point.

<a id="comparison"></a>

## 5. The comparison: learned representation vs. simple baseline

Yesterday's probe showed the embeddings contain localization information.  Today's sharper question: **is that information anything more than what a trivial representation would carry?**

The baseline: represent each protein by its **amino-acid composition** — 20 numbers, the frequency of each residue type.  No model, no training, computable with `str.count`.  Then run the *same* probe (same classifier, same grouped split by `uniref50` family) on both representations of the same proteins.  Only the representation changes:

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_score

y = proteins['location'].to_numpy()
families = proteins['uniref50'].to_numpy()

AAS = 'ACDEFGHIKLMNPQRSTVWY'
composition = np.zeros((len(proteins), 20))
for i, s in enumerate(proteins['sequence']):
    counts = np.array([s.count(a) for a in AAS], float)
    composition[i] = counts / counts.sum()

clf = LogisticRegression(max_iter=2000)
for name, X in [('amino-acid composition (20 numbers)', composition),
                ('ESM-2 embedding (320 numbers)', embeddings)]:
    acc = cross_val_score(clf, X, y, cv=GroupKFold(5), groups=families).mean()
    print(f'{name:38s} grouped CV accuracy = {acc:.3f}')
print(f'{"chance":38s} = 0.333')

amino-acid composition (20 numbers)    grouped CV accuracy = 0.642
ESM-2 embedding (320 numbers)          grouped CV accuracy = 0.902
chance                                 = 0.333


The performance difference here is the scientific finding of the session:

* Composition scores about **0.64** — well above chance.  That is respectable: secreted and membrane proteins have characteristic compositions, and a 20-number representation captures some of it.
* The ESM-2 representation scores about **0.90** — on a split where whole protein families are held out together.  The extra ~26 points are information the model learned about how proteins work, from **unlabeled sequences alone**. Nobody ever told it what "secreted" means, or that localization exists.

That is the foundation-model use case. Someone else paid for the training. The marginal cost of the knowledge we add to it with this analysis is a few seconds of laptop time.

<a id="residue"></a>

## 6. Per-residue embeddings and attention

Two things worth knowing exist one level below the sequence embeddings used today:

* **Per-residue embeddings.**  The mean-pooling step collapsed `last_hidden_state` from one-vector-per-residue to one-vector-per-sequence.  For questions about *positions* — which residue is modified, where a binding site is, per-base effects in DNA models — skip the pooling and use the per-position vectors directly.  Same extraction, one line less.
* **Attention maps.**  The model's internal attention weights can be read out, and for protein models they weakly predict which residues contact each other in the folded structure.  This is interesting — but how interesting is debated. Some argue that attention maps are widely **oversold as "explanations"** of what a model is doing, and that they are not that.  This afternoon's session returns to the question of what can and cannot be concluded from a model's internals.

<a id="homework"></a>

## Homework

Three functions in `a14am_hw2.py`, which together are the entire reusable workflow.  As before: open the file, replace each `raise NotImplementedError` line, and use the check cells.  `embed_sequences` is for working on here in session; `composition_baseline` and `probe` complete the set — and `probe` is *yesterday's `linear_probe` with a new name*: reuse the solution.  One probe, any representation.

In [ ]:
import a14am_hw2, importlib
importlib.reload(a14am_hw2)
help(a14am_hw2)

**Check `embed_sequences`.**  The strongest possible check: it should reproduce the shipped `protein_embeddings.npy` to numerical precision (tested on the first 32 sequences for speed).  Expected output: `reproduces the shipped file: True`

In [ ]:
importlib.reload(a14am_hw2)

if MODEL_AVAILABLE:
    mine = a14am_hw2.embed_sequences(model, tokenizer, list(proteins['sequence'][:32]))
    print('shape:', mine.shape)
    print('reproduces the shipped file:', np.allclose(mine, shipped_embeddings[:32], atol=1e-4))
else:
    print('(model unavailable -- this check needs the live model; try it after the session)')

**Check `composition_baseline`.**  Expected output:

```
shape: (450, 20)   rows sum to 1: True
```

In [ ]:
importlib.reload(a14am_hw2)

Xc = a14am_hw2.composition_baseline(list(proteins['sequence']))
print('shape:', Xc.shape, '  rows sum to 1:', np.allclose(Xc.sum(axis=1), 1))

**Check `probe`.**  Run on the representations computed in Section 5, so it works even before the other two functions are done.  Expected output (matching Section 5 and yesterday's homework):

```
composition: 0.642   embeddings: 0.902
```

In [ ]:
importlib.reload(a14am_hw2)

pc = a14am_hw2.probe(composition, y, groups=families)
pe = a14am_hw2.probe(embeddings, y, groups=families)
print(f'composition: {pc:.3f}   embeddings: {pe:.3f}')

<a id="recipe"></a>

## Summary

For any data type, the workflow practiced today is:

1. **Find a model** on the hub for the relevant data or modality (the Section 1 table is a starting map; larger checkpoints of ESM-2 exist too, same recipe).
2. **Read the model card** — what it was trained on, what its authors say it is for, and any usage restrictions.  Training data matters scientifically: if our own data resembles it too closely, this afternoon's session explains why that can quietly corrupt an evaluation.
3. **Extract frozen representations** on a CPU (`embed_sequences` generalizes with minor edits — mostly the tokenizer call).
4. **Probe** for the variable of interest (`probe` needs no edits at all).
5. **Validate honestly** — grouped splits, no leakage, the Thursday-morning checklist.

What can go wrong with step 5 even when everything above was done right is the subject of the final session, this afternoon.